In [1]:
import pandas as pd

df = pd.read_csv('heart_disease_uci.csv')

# num is 0-4 severity. For a first model, let's simplify to yes/no.
df['has_disease'] = (df['num'] > 0).astype(int)

df[['num', 'has_disease']].head(10)

,num,has_disease
0,0,0
1,2,1
2,1,1
3,0,0
4,0,0
5,0,0
6,3,1
7,0,0
8,2,1
9,1,1


In [2]:
features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak']

X = df[features].copy()
y = df['has_disease']

X.isnull().sum()

age          0
sex          0
cp           0
trestbps    59
chol        30
fbs         90
restecg      2
thalch      55
exang       55
oldpeak     62
dtype: int64

In [3]:
for col in ['trestbps', 'chol', 'thalch', 'oldpeak']:
    X[col] = X[col].fillna(X[col].median())

for col in ['fbs', 'restecg', 'exang']:
    X[col] = X[col].fillna(X[col].mode()[0])

X.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
dtype: int64

In [4]:
X.dtypes

age           int64
sex             str
cp              str
trestbps    float64
chol        float64
fbs          object
restecg         str
thalch      float64
exang        object
oldpeak     float64
dtype: object

In [5]:
X = pd.get_dummies(X, columns=['sex', 'cp', 'fbs', 'restecg', 'exang'], drop_first=True)

X.dtypes

age                           int64
trestbps                    float64
chol                        float64
thalch                      float64
oldpeak                     float64
sex_Male                       bool
cp_atypical angina             bool
cp_non-anginal                 bool
cp_typical angina              bool
fbs_True                       bool
restecg_normal                 bool
restecg_st-t abnormality       bool
exang_True                     bool
dtype: object

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((736, 13), (184, 13))

In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Training complete")

Training complete


In [9]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2%}")

print(classification_report(y_test, predictions))

Accuracy: 80.43%
              precision    recall  f1-score   support

           0       0.75      0.79      0.77        75
           1       0.85      0.82      0.83       109

    accuracy                           0.80       184
   macro avg       0.80      0.80      0.80       184
weighted avg       0.81      0.80      0.81       184



In [10]:
import pandas as pd

coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})

coefficients = coefficients.sort_values('coefficient', ascending=False)
coefficients

,feature,coefficient
5,sex_Male,1.427891
12,exang_True,0.940244
4,oldpeak,0.603243
9,fbs_True,0.420114
11,restecg_st-t abnormality,0.121941
0,age,0.022609
1,trestbps,0.002079
2,chol,-0.004132
3,thalch,-0.012613
10,restecg_normal,-0.046378


In [11]:
probabilities = model.predict_proba(X_test)[:, 1]  # probability of "has disease"

probabilities[:10]

array([0.08528761, 0.22872859, 0.89115013, 0.90392063, 0.28797908,
       0.11634867, 0.15245803, 0.97999008, 0.84116798, 0.2238524 ])

In [12]:
from sklearn.metrics import classification_report

threshold = 0.35
predictions_adjusted = (probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions_adjusted))

              precision    recall  f1-score   support

           0       0.80      0.73      0.76        75
           1       0.83      0.87      0.85       109

    accuracy                           0.82       184
   macro avg       0.81      0.80      0.81       184
weighted avg       0.81      0.82      0.81       184

